# Sparsity–Entanglement Phase Diagram

Maps circuit families onto the (PR/2^N, S/S_max) plane. Confidence ellipses
(95 %, 2-σ covariance) show the per-family uncertainty regions.


In [ ]:
import sys, json, numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from scipy.stats import gaussian_kde

from src.experimentation.runner import (
    run_trial_timed,
    exact_statevector,
)


def find_repo_root(start=None):
    start = (
        __import__("pathlib").Path.cwd()
        if start is None
        else __import__("pathlib").Path(start).resolve()
    )
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

THIS_DIR = REPO_ROOT / "tests"
FIGURE_DIR = THIS_DIR / "figures"
DATA_DIR = THIS_DIR / "results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

from src.experimentation.runner import (
    exact_statevector,
    make_rfim,
    make_brickwork,
    make_brickwork_2d,
    make_haar,
    make_qaoa,
    make_uccsd,
    make_tfim,
    confidence_ellipse,
)
from src.visualization.style import mplstyle, figax, polish_axes, save_figure, COLORS

mplstyle(dpi=180)
mpl.rcParams.update({"font.size": 9.0})

In [ ]:
N = 16
K = 4096
N_TRIALS = 20
SEED = 0
BASS_KW = dict(optimize_every=5, buffer_factor=1)
FORCE_RERUN = True

CIRCUIT_FAMILIES = {
    "RFIM (W=2)": {
        "color": "#1C875C",
        "marker": "D",
        "factory": lambda N, rng: make_rfim(N, rng, layers=5, W=2.0, dt=0.2),
    },
    "Brickwork 1D": {
        "color": "#0C5DA5",
        "marker": "s",
        "factory": lambda N, rng: make_brickwork(N, depth=6, rng=rng),
    },
    "Brickwork 2D": {
        "color": "#8E6DB0",
        "marker": "^",
        "factory": lambda N, rng: make_brickwork_2d(rows=4, cols=4, depth=4, rng=rng),
    },
    "Haar L=3": {
        "color": "#E87722",
        "marker": "o",
        "factory": lambda N, rng: make_haar(N, depth=3, rng=rng),
    },
    "QAOA p=3": {
        "color": "#B3446C",
        "marker": "P",
        "factory": lambda N, rng: make_qaoa(N, rounds=3, rng=rng),
    },
    "UCCSD L=1": {
        "color": "#888888",
        "marker": "v",
        "factory": lambda N, rng: make_uccsd(N, rng, n_trotter_steps=1),
    },
}

In [ ]:
def bipartite_entropy(sv, N):
    cut = N // 2
    psi = np.asarray(sv).reshape(2**cut, 2 ** (N - cut))
    rho = psi @ psi.conj().T
    ev = np.linalg.eigvalsh(rho)
    ev = ev[ev > 1e-15]
    return float(-np.sum(ev * np.log2(ev)))


def exact_pr(sv):
    p = np.abs(np.asarray(sv)) ** 2
    return float(1.0 / np.sum(p**2)) if np.sum(p**2) > 0 else np.nan


S_max = float(N // 2)  # maximum entropy = N/2 ebits (half chain)

_npz = DATA_DIR / "phase_diagram.npz"
_meta = DATA_DIR / "phase_diagram_meta.json"
_exp = {
    "N": N,
    "K": K,
    "n_trials": N_TRIALS,
    "seed": SEED,
    "families": list(CIRCUIT_FAMILIES.keys()),
}

if _npz.exists() and _meta.exists() and not FORCE_RERUN:
    stored = json.loads(_meta.read_text())
    if stored == _exp:
        raw = dict(np.load(_npz, allow_pickle=False))
        pr_norm = raw["pr_norm"]
        entropy = raw["entropy"]
        print(f"Loaded cache: {_npz}")
else:
    n_fam = len(CIRCUIT_FAMILIES)
    pr_norm = np.full((n_fam, N_TRIALS), np.nan)
    entropy = np.full((n_fam, N_TRIALS), np.nan)
    for fi, (label, fam) in enumerate(CIRCUIT_FAMILIES.items()):
        print(f"[{label}]")
        for t in range(N_TRIALS):
            seed_t = SEED * 10**6 + fi * 1000 + t
            rng = np.random.default_rng(seed_t)
            circ = fam["factory"](N, rng)
            sv = exact_statevector(N, circ)
            pr_norm[fi, t] = exact_pr(sv) / 2**N
            entropy[fi, t] = bipartite_entropy(sv, N) / S_max
            print(
                f"  trial {t}: PR/2^N={pr_norm[fi,t]:.4f}  S/S_max={entropy[fi,t]:.3f}"
            )
    np.savez_compressed(_npz, pr_norm=pr_norm, entropy=entropy)
    _meta.write_text(json.dumps(_exp, indent=2))
    print(f"Saved {_npz}")

print("\n--- Phase Diagram Summary ---")
hdr = f"{'Family':<18}  {'PR/2^N med[IQR]':<30}  {'S/S_max med[IQR]':<28}  {'n'}"
print(hdr)
print("-" * len(hdr))
for fi, label in enumerate(CIRCUIT_FAMILIES):
    pr = pr_norm[fi, :][np.isfinite(pr_norm[fi, :])]
    ent = entropy[fi, :][np.isfinite(entropy[fi, :])]

    def fmt_iqr(d):
        p25, med, p75 = np.percentile(d, [25, 50, 75])
        return f"{med:.4f} [{p25:.4f},{p75:.4f}]"

    print(f"{label:<18}  {fmt_iqr(pr):<30}  {fmt_iqr(ent):<28}  {len(pr)}")

In [ ]:
# SETTINGS
EXT_CACHE = DATA_DIR / "phase_diagram_extended.npz"
EXT_META = DATA_DIR / "phase_diagram_extended_meta.json"

S_max = float(N // 2)

# ENTANGLEMENT


def bipartite_entropy(sv, N):

    cut = N // 2

    psi = np.asarray(sv).reshape(
        2**cut,
        2 ** (N - cut),
    )

    rho = psi @ psi.conj().T

    ev = np.linalg.eigvalsh(rho)

    ev = ev[ev > 1e-15]

    return float(-np.sum(ev * np.log2(ev))) / S_max


# ARRAYS

n_fam = len(CIRCUIT_FAMILIES)

shape = (n_fam, N_TRIALS)

# EXACT

pr_exact = np.full(shape, np.nan)
entropy_exact = np.full(shape, np.nan)

# FIXED / TRUSTS

fidelity_fixed = np.full(shape, np.nan)
pr_fixed = np.full(shape, np.nan)

# BASS

fidelity_bass_arr = np.full(shape, np.nan)
pr_bass = np.full(shape, np.nan)

# runtime

runtime_fixed = np.full(shape, np.nan)
runtime_bass = np.full(shape, np.nan)

# LOAD OLD EXACT CACHE

old_npz = DATA_DIR / "phase_diagram.npz"

if old_npz.exists():

    raw = np.load(old_npz)

    if "pr_norm" in raw:
        pr_exact[:] = raw["pr_norm"]

    if "entropy" in raw:
        entropy_exact[:] = raw["entropy"]

    print(f"Loaded existing exact cache: {old_npz}")

# MAIN LOOP

for fi, (label, fam) in enumerate(CIRCUIT_FAMILIES.items()):

    print("\n================================================")
    print(label)
    print("================================================")

    # SKIP UCCSD

    skip_exact = "uccsd" in label.lower()

    if skip_exact:
        print("Skipping expensive UCCSD rerun.")
        print("Keeping cached exact coordinates.")
        continue

    for t in range(N_TRIALS):

        seed_t = SEED * 10**6 + fi * 1000 + t

        rng = np.random.default_rng(seed_t)

        circuit = fam["factory"](N, rng)

        # EXACT

        if not skip_exact:

            exact_sv = exact_statevector(
                N,
                circuit,
            )

            # entropy for exact state
            entropy_exact[fi, t] = bipartite_entropy(
                exact_sv,
                N,
            )

        else:

            exact_sv = None

        # FIXED + BASS

        res = run_trial_timed(
            circuit_gen=fam["factory"],
            N=N,
            k=K,
            seed=seed_t,
            bass_kwargs={},
            _exact_sv=exact_sv,
        )

        # fixedbasis

        fidelity_fixed[fi, t] = res["f_fixed"]

        pr_fixed[fi, t] = res["pr_fixed"] / (2**N)

        runtime_fixed[fi, t] = res["t_fixed"]

        # BASS

        fidelity_bass_arr[fi, t] = res["f_bass"]

        pr_bass[fi, t] = res["pr_bass"] / (2**N)

        runtime_bass[fi, t] = res["t_bass"]

        # PRINT

        print(
            f"trial {t:02d} | "
            f"PR_exact={pr_exact[fi,t]:.4e} | "
            f"PR_fixed={pr_fixed[fi,t]:.4e} | "
            f"PR_BASS={pr_bass[fi,t]:.4e} | "
            f"F_fix={fidelity_fixed[fi,t]:.4f} | "
            f"F_BASS={fidelity_bass_arr[fi,t]:.4f}"
        )

# SAVE CACHE

np.savez_compressed(
    EXT_CACHE,
    # exact
    pr_exact=pr_exact,
    entropy_exact=entropy_exact,
    # fixed
    fidelity_fixed=fidelity_fixed,
    pr_fixed=pr_fixed,
    # bass
    fidelity_bass=fidelity_bass_arr,
    pr_bass=pr_bass,
    # runtimes
    runtime_fixed=runtime_fixed,
    runtime_bass=runtime_bass,
)

meta = dict(
    N=N,
    K=K,
    trials=N_TRIALS,
    seed=SEED,
    families=list(CIRCUIT_FAMILIES.keys()),
)

EXT_META.write_text(json.dumps(meta, indent=2))

print("\n================================================")
print("SAVED EXTENDED CACHE")
print("================================================")
print(EXT_CACHE)

# UCCSD PRINT FUNCTION


def print_uccsd_data():

    idx = None

    for i, label in enumerate(CIRCUIT_FAMILIES.keys()):

        if "uccsd" in label.lower():

            idx = i
            break

    if idx is None:

        print("UCCSD not found.")
        return

    print("\n================================================")
    print(" UCCSD DATA")
    print("================================================")

    for t in range(N_TRIALS):

        print(
            f"trial {t:02d} | "
            f"PR_exact={pr_exact[idx,t]:.6e} | "
            f"S_exact={entropy_exact[idx,t]:.6f}"
        )

    print("\nSUMMARY")

    print(f"median PR : " f"{np.nanmedian(pr_exact[idx]):.6e}")

    print(f"median S  : " f"{np.nanmedian(entropy_exact[idx]):.6f}")

    print(
        f"PR range  : "
        f"{np.nanmin(pr_exact[idx]):.6e} -> "
        f"{np.nanmax(pr_exact[idx]):.6e}"
    )

    print(
        f"S range   : "
        f"{np.nanmin(entropy_exact[idx]):.6f} -> "
        f"{np.nanmax(entropy_exact[idx]):.6f}"
    )


# OPTIONAL

print_uccsd_data()

In [ ]:
# PHASE DIAGRAM

from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import numpy as np

fig, ax = figax(w=5, h=4)

# AXES

ax.set_xscale("log")

ax.set_xlim(1e-4, 1)
ax.set_ylim(0.0, 1.1)

ax.tick_params(
    axis="both",
    which="major",
    width=1.2,
    length=6,
)

ax.tick_params(
    axis="both",
    which="minor",
    width=1.0,
    length=3,
)

# PHASE REGIONS

boundary_x = float(np.nanmedian(pr_norm))

ax.axvspan(
    1e-4,
    boundary_x,
    ymin=0.0,
    ymax=0.5,
    alpha=0.08,
    color="#90CAF9",
)

ax.axvspan(
    1e-4,
    boundary_x,
    ymin=0.5,
    ymax=1.0,
    alpha=0.08,
    color="#EF9A9A",
)

ax.axvspan(
    boundary_x,
    1.0,
    ymin=0.0,
    ymax=0.5,
    alpha=0.08,
    color="#A5D6A7",
)

ax.axvspan(
    boundary_x,
    1.0,
    ymin=0.5,
    ymax=1.0,
    alpha=0.08,
    color="#CE93D8",
)

# REGION LABELS

ax.text(
    3e-4,
    0.23,
    "Sparse-friendly\nLow entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    3e-4,
    0.73,
    "Structured many-body\nHigh entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    0.05,
    0.13,
    "Dense superposition\nWeak entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    0.18,
    0.83,
    "Chaotic / Haar-like\nVolume-law",
    fontsize=8,
    fontweight="bold",
    ha="center",
    alpha=0.75,
)

# SCATTER

handles = []

ALL_X = []
ALL_Y = []

for fi, (label, fam) in enumerate(CIRCUIT_FAMILIES.items()):

    pr_fam = pr_norm[fi, :]
    ent_fam = entropy[fi, :]

    mask = np.isfinite(pr_fam) & np.isfinite(ent_fam) & (pr_fam > 0)

    pr_fam = pr_fam[mask]
    ent_fam = ent_fam[mask]

    c = fam["color"]
    mk = fam["marker"]

    ALL_X.extend(pr_fam.tolist())
    ALL_Y.extend(ent_fam.tolist())

    # individual samples

    ax.scatter(
        pr_fam,
        ent_fam,
        s=65,
        marker=mk,
        color=c,
        edgecolor="white",
        linewidth=0.9,
        alpha=0.72,
        zorder=5,
    )

    # family centroid

    ax.scatter(
        np.mean(pr_fam),
        np.mean(ent_fam),
        s=110,
        marker=mk,
        color=c,
        edgecolor="black",
        linewidth=1.4,
        zorder=7,
    )

    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker=mk,
            color="w",
            markerfacecolor=c,
            markeredgecolor="black",
            markersize=6,
            linewidth=0,
            label=label,
        )
    )

# KDE CONTOURS

ALL_X = np.array(ALL_X)
ALL_Y = np.array(ALL_Y)

if len(ALL_X) > 10:

    kde = gaussian_kde(
        np.vstack(
            [
                np.log10(ALL_X),
                ALL_Y,
            ]
        ),
        bw_method=0.42,
    )

    # IMPORTANT:
    # use axis limits directly
    xi = np.linspace(
        np.log10(1e-4),
        0,
        240,
    )

    yi = np.linspace(
        0,
        1.05,
        240,
    )

    Xi, Yi = np.meshgrid(
        xi,
        yi,
    )

    Zi = kde(
        np.vstack(
            [
                Xi.ravel(),
                Yi.ravel(),
            ]
        )
    ).reshape(Xi.shape)

    ax.contour(
        10**Xi,
        Yi,
        Zi,
        levels=6,
        linewidths=1.5,
        alpha=0.32,
        colors="black",
        zorder=2,
    )

# BOUNDARY

ax.axvline(
    boundary_x,
    ls="--",
    lw=1.8,
    color="black",
    alpha=0.7,
    zorder=3,
)

ax.text(
    boundary_x * 1.03,
    0.98,
    r"median PR/$2^N$",
    fontsize=8,
    alpha=0.75,
)

# LABELS

ax.set_xlabel(
    r"Normalized participation ratio $\mathrm{PR}/2^N$",
    fontsize=10,
)

ax.set_ylabel(
    r"Normalized entanglement entropy $S/S_{\max}$",
    fontsize=10,
)

# LEGEND

ax.legend(handles=handles, loc="upper left", frameon=False, ncol=2, fontsize=8)

# GRID / STYLING

ax.grid(
    True,
    which="major",
    linestyle=":",
    alpha=0.22,
)

polish_axes(ax)

# SAVE

save_figure(
    fig,
    FIGURE_DIR / "fig_phase_diagram",
)

plt.show()

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = figax(w=5, h=4)

ax.set_xscale("log")

ax.set_xlim(1e-4, 1)
ax.set_ylim(0.0, 1.1)

ax.tick_params(
    axis="both",
    which="major",
    width=1.2,
    length=6,
)

ax.tick_params(
    axis="both",
    which="minor",
    width=1.0,
    length=3,
)

boundary_x = float(np.nanmedian(pr_exact))

regions = [
    ((1e-4, boundary_x), (0.0, 0.5), "#90CAF9"),
    ((1e-4, boundary_x), (0.5, 1.0), "#EF9A9A"),
    ((boundary_x, 1.0), (0.0, 0.5), "#A5D6A7"),
    ((boundary_x, 1.0), (0.5, 1.0), "#CE93D8"),
]

for (xmin, xmax), (ymin, ymax), color in regions:

    ax.axvspan(
        xmin,
        xmax,
        ymin=ymin,
        ymax=ymax,
        alpha=0.08,
        color=color,
    )

ax.text(
    3e-4,
    0.23,
    "Sparse-friendly\nLow entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    3e-4,
    0.73,
    "Structured many-body\nHigh entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    0.05,
    0.13,
    "Dense superposition\nWeak entanglement",
    fontsize=8,
    fontweight="bold",
    alpha=0.75,
)

ax.text(
    0.18,
    0.83,
    "Chaotic / Haar-like\nVolume-law",
    fontsize=8,
    fontweight="bold",
    ha="center",
    alpha=0.75,
)

handles = []

ALL_X = []
ALL_Y = []

for fi, (label, fam) in enumerate(CIRCUIT_FAMILIES.items()):

    pr_fam = pr_exact[fi]
    ent_fam = entropy_exact[fi]

    mask = np.isfinite(pr_fam) & np.isfinite(ent_fam) & (pr_fam > 0)

    pr_fam = pr_fam[mask]
    ent_fam = ent_fam[mask]

    c = fam["color"]
    mk = fam["marker"]

    ALL_X.extend(pr_fam.tolist())
    ALL_Y.extend(ent_fam.tolist())

    arrow_indices = []

    if "uccsd" not in label.lower():

        shifts = []

        for j in range(len(pr_fam)):

            x0 = pr_exact[fi, j]
            x1 = pr_bass[fi, j]

            if np.isfinite(x0) and np.isfinite(x1) and x0 > 0 and x1 > 0:

                d = abs(np.log10(x1) - np.log10(x0))

                shifts.append((d, j))

        shifts = sorted(
            shifts,
            reverse=True,
        )

        arrow_indices = [j for _, j in shifts[:4]]

    for j in range(len(pr_fam)):

        x_exact = pr_exact[fi, j]
        y_exact = entropy_exact[fi, j]

        ax.scatter(
            x_exact,
            y_exact,
            s=65,
            marker=mk,
            color=c,
            edgecolor="white",
            linewidth=0.9,
            alpha=0.72,
            zorder=5,
        )

        if j not in arrow_indices:
            continue

        x_end = pr_bass[fi, j]

        if not np.isfinite(x_end) or x_end <= 0:
            continue

        delta_logx = abs(np.log10(x_end) - np.log10(x_exact))

        if delta_logx < 0.08:
            continue

        ax.annotate(
            "",
            xy=(x_end, y_exact),
            xytext=(x_exact, y_exact),
            arrowprops=dict(
                arrowstyle="->",
                lw=1.2,
                color=c,
                alpha=0.55,
                shrinkA=0,
                shrinkB=0,
                mutation_scale=10,
                connectionstyle="arc3,rad=0.08",
            ),
            zorder=4,
        )

    ax.scatter(
        np.mean(pr_fam),
        np.mean(ent_fam),
        s=110,
        marker=mk,
        color=c,
        edgecolor="black",
        linewidth=1.4,
        zorder=7,
    )

    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker=mk,
            color="w",
            markerfacecolor=c,
            markeredgecolor="black",
            markersize=6,
            linewidth=0,
            label=label,
        )
    )

ALL_X = np.array(ALL_X)
ALL_Y = np.array(ALL_Y)

if len(ALL_X) > 10:

    kde = gaussian_kde(
        np.vstack(
            [
                np.log10(ALL_X),
                ALL_Y,
            ]
        ),
        bw_method=0.42,
    )

    xi = np.linspace(
        np.log10(1e-4),
        0,
        240,
    )

    yi = np.linspace(
        0,
        1.05,
        240,
    )

    Xi, Yi = np.meshgrid(
        xi,
        yi,
    )

    Zi = kde(
        np.vstack(
            [
                Xi.ravel(),
                Yi.ravel(),
            ]
        )
    ).reshape(Xi.shape)

    ax.contour(
        10**Xi,
        Yi,
        Zi,
        levels=6,
        linewidths=1.5,
        alpha=0.32,
        colors="black",
        zorder=2,
    )

ax.axvline(
    boundary_x,
    ls="--",
    lw=1.8,
    color="black",
    alpha=0.7,
    zorder=3,
)

ax.text(
    boundary_x * 1.03,
    0.98,
    r"median PR/$2^N$",
    fontsize=8,
    alpha=0.75,
)

ax.set_xlabel(
    r"Normalized participation ratio $\mathrm{PR}/2^N$",
    fontsize=10,
)

ax.set_ylabel(
    r"Normalized entanglement entropy $S/S_{\max}$",
    fontsize=10,
)

ax.legend(
    handles=handles,
    loc="upper left",
    frameon=False,
    ncol=2,
    fontsize=8,
)

ax.grid(
    True,
    which="major",
    linestyle=":",
    alpha=0.22,
)

polish_axes(ax)

save_figure(
    fig,
    FIGURE_DIR / "fig_phase_diagram",
)

plt.show()